# NB_patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the Silver-to-Gold STTM. Full rebuild (CREATE OR REPLACE) with LEFT JOINs from patients.

In [0]:
# Parameterisation disabled: catalog is hard-coded as adb_caresync in the SQL cells below
# # Unity Catalog catalog name (STTM placeholder <catalog>)
# dbutils.widgets.text("catalog", "adb_caresync")
# print("catalog =", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

CREATE OR REPLACE TABLE adb_caresync.gold.patient_360 AS
WITH lab_agg AS (
  -- Aggregate lab results to patient grain
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  CAST(p.first_name AS STRING)                                           AS first_name,
  CAST(p.last_name AS STRING)                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                               AS gender,
  CAST(p.city AS STRING)                                                 AS city,
  CAST(p.state AS STRING)                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                       AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                               AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                       AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                       AS critical_test_count,
  CAST(la.last_test_date AS DATE)                                        AS last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:
%sql
-- Grain check: row count must equal silver.patients, patient_id must be unique
SELECT
  (SELECT COUNT(*) FROM adb_caresync.silver.patients) AS source_patients,
  COUNT(*)                                   AS gold_rows,
  COUNT(DISTINCT patient_id)                 AS distinct_patients,
  SUM(total_tests)                           AS sum_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS source_lab_results,
  COUNT_IF(hospital_name IS NULL)            AS missing_hospital_name,
  COUNT_IF(insurance_provider_name IS NULL)  AS missing_insurance_name,
  COUNT_IF(total_tests = 0)                  AS patients_without_tests
FROM adb_caresync.gold.patient_360;

In [0]:
%sql
SELECT * FROM adb_caresync.gold.patient_360 ORDER BY patient_id LIMIT 20;

In [0]:
updated_df = spark.table("adb_caresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))